In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import urllib.request, json
pd.set_option('display.max_columns', None)


import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'EPA')

path_code    = os.path.join(path_git, 'Data', 'EPA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

***

Monthly levels of concern

***

In [ ]:
indicator_name = 'Health_3'

file_name = f"{indicator_name} MSA EPA.xlsx"
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MSA')

display(df_msa.head())

In [ ]:
# Set Indicator
indicator_name = 'Health_3'
plot_name = 'aqi_monthly'
export = False


## Organizing ---


df_plot = df_msa.copy()

df_plot['date_local'] = pd.to_datetime(df_plot['date_local'])
df_plot['Month'] = df_plot['date_local'].dt.month
df_plot['Year' ] = df_plot['date_local'].dt.year
df_plot['date_local'] = df_plot['date_local'].apply(lambda x: x.strftime('%Y-%m'))
df_plot = df_plot[df_plot['Year'] >= 1999]


df_plot = pd.DataFrame(df_plot[['MSA', 'date_local', 'Level of Concern']].value_counts())
df_plot = df_plot.reset_index()

list_aqi = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
df_plot['Sort'] = pd.Categorical(df_plot['Level of Concern'], list_aqi)
df_plot = df_plot.sort_values(['MSA', 'date_local', 'Sort'], ascending = [True, False, True])
df_plot = df_plot.drop('Sort', axis=1)
df_plot = df_plot[df_plot['MSA'] == 'Sacramento--Roseville--Arden-Arcade, CA']

df_plot = df_plot.reset_index(drop=True)

display(df_plot.head())


## Plotting ---


color_map  = {
    'Good': '#00FF00'
    , 'Moderate': '#FFFF00'
    , 'Unhealthy for Sensitive Groups': '#FFA500'
    , 'Unhealthy': '#FF0000'
    , 'Very Unhealthy': '#800080'
    , 'Hazardous': '#800000'
}


fig = px.bar(df_plot, x='date_local', y='count', color='Level of Concern', color_discrete_map=color_map)


title = '<b>Air Quality Index Levels of Concern (Ozone and/or PM2.5)</b>  <br><sup>4-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=5, range = [0, 31])
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")
fig.update_traces(hovertemplate='Number of days: %{y}<br>%{x}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Annual levels of concern

***

In [ ]:
indicator_name = 'Health_3'

file_name = f"{indicator_name} MSA EPA.xlsx"
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MSA')

display(df_msa.head())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

# Set Indicator
indicator_name = 'Health_3'
plot_name = 'aqi_annual'
export = False


## Organizing ---


df_plot = df_msa.copy()

df_plot['date_local'] = pd.to_datetime(df_plot['date_local'])
df_plot['Year' ] = df_plot['date_local'].dt.year


df_plot = pd.DataFrame(df_plot[['MSA', 'Year', 'Level of Concern']].value_counts())
df_plot = df_plot.reset_index()

list_aqi = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
df_plot['Sort'] = pd.Categorical(df_plot['Level of Concern'], list_aqi)

df_plot = df_plot.sort_values(['MSA', 'Year', 'Sort'], ascending = [True, False, True])
df_plot = df_plot.drop('Sort', axis=1)
df_plot = df_plot[df_plot['MSA'] == 'Sacramento--Roseville--Arden-Arcade, CA']
df_plot = df_plot[df_plot['Year'] >= 1999]

df_plot = df_plot.reset_index()

display(df_plot.head())


## Plotting ---


color_map  = {
    'Good': '#00FF00'
    , 'Moderate': '#FFFF00'
    , 'Unhealthy for Sensitive Groups': '#FFA500'
    , 'Unhealthy': '#FF0000'
    , 'Very Unhealthy': '#800080'
    , 'Hazardous': '#800000'
}


fig = px.bar(df_plot, x='Year', y='count', color='Level of Concern', color_discrete_map=color_map)


title = '<b>Air Quality Index Levels of Concern (Ozone and/or PM2.5)</b>  <br><sup>4-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=50, range = [0, 365])
fig.update_xaxes(tick0=0, dtick=4, range=[1998.5, 2023.5])
fig.update_traces(hovertemplate='Number of days: %{y}<br>%{x}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)